In [0]:
import urllib.request, zipfile, os

# Where we'll stage the raw file. /tmp is local to the driver and fine for this.
url = "https://s3.amazonaws.com/tripdata/202412-citibike-tripdata.zip"
zip_path = "/tmp/citibike.zip"
extract_dir = "/tmp/citibike_csv"

# Download
urllib.request.urlretrieve(url, zip_path)
print("Downloaded:", os.path.getsize(zip_path), "bytes")

# Unzip (handles single- or multi-CSV archives)
os.makedirs(extract_dir, exist_ok=True)
with zipfile.ZipFile(zip_path) as z:
    z.extractall(extract_dir)

# Show what came out
for root, _, files in os.walk(extract_dir):
    for f in files:
        print(os.path.join(root, f))

In [0]:
catalog = "workspace"
schema  = "default"
volume  = "citibike"

# Create the volume if it doesn't exist
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.{schema}.{volume}")

volume_path = f"/Volumes/{catalog}/{schema}/{volume}"
print("Volume path:", volume_path)

# Copy the extracted CSVs from local /tmp into the volume
import shutil, glob, os
for src in glob.glob("/tmp/citibike_csv/*.csv"):
    dst = os.path.join(volume_path, os.path.basename(src))
    shutil.copy(src, dst)
    print("Copied:", dst)

# Confirm what's in the volume
display(dbutils.fs.ls(volume_path))

In [0]:
volume_path = "/Volumes/workspace/default/citibike"

df = (spark.read
      .format("csv")
      .option("header", "true")
      .option("inferSchema", "true")
      .load(f"{volume_path}/202412-citibike-tripdata_*.csv"))   # wildcard reads all 3

print("Row count:", df.count())
df.printSchema()
df.show(5, truncate=False)

In [0]:
from pyspark.sql.functions import col

top_ten = (df
    .filter(col("start_station_id").isNotNull())
    .groupBy("start_station_id", "start_station_name")
    .count()
    .orderBy(col("count").desc())
    .limit(10))

top_ten.show(truncate=False)

In [0]:
table_name = "workspace.default.citibike_top_ten_start_stations"

(top_ten.write
    .format("delta")          # explicit, though Delta is the default on Databricks
    .mode("overwrite")
    .saveAsTable(table_name))

print("Wrote Delta table:", table_name)

# Confirm it's a real, queryable managed table
spark.sql(f"DESCRIBE DETAIL {table_name}").select("format", "location", "numFiles").show(truncate=False)

In [0]:
from delta.tables import DeltaTable

table_name = "workspace.default.citibike_top_ten_start_stations"

# Simulate an update batch: one existing station with a corrected count,
# and one brand-new station not currently in the table.
updates = spark.createDataFrame(
    [
        ("6140.05", "W 21 St & 6 Ave", 9999),   # existing id -> will UPDATE
        ("9999.99", "Test Station (new)", 5000), # new id    -> will INSERT
    ],
    ["start_station_id", "start_station_name", "count"]
)

delta_tbl = DeltaTable.forName(spark, table_name)

(delta_tbl.alias("t")
    .merge(updates.alias("s"), "t.start_station_id = s.start_station_id")
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute())

spark.sql(f"SELECT * FROM {table_name} ORDER BY count DESC").show(truncate=False)

In [0]:
table_name = "workspace.default.citibike_top_ten_start_stations"

# Show the transaction log: every version, with the operation that created it
spark.sql(f"DESCRIBE HISTORY {table_name}") \
     .select("version", "timestamp", "operation") \
     .show(truncate=False)

# Query the table AS IT WAS at version 0 — before the MERGE
print("=== Version 0 (original, pre-merge) ===")
spark.sql(f"SELECT * FROM {table_name} VERSION AS OF 0 ORDER BY count DESC").show(truncate=False)

In [0]:
table_name = "workspace.default.citibike_top_ten_start_stations"

# Compact small files; Z-order by the column you'd most often filter/join on
spark.sql(f"OPTIMIZE {table_name} ZORDER BY (start_station_id)").show(truncate=False)

# OPTIMIZE itself creates a new version in the log
spark.sql(f"DESCRIBE HISTORY {table_name}").select("version", "operation").show(truncate=False)

In [0]:
%sql
-- Top 5 stations by trips, querying the Delta table directly in SQL
SELECT
  start_station_id,
  start_station_name,
  count AS trip_count
FROM workspace.default.citibike_top_ten_start_stations
WHERE start_station_id != '9999.99'        -- exclude the synthetic test row
ORDER BY trip_count DESC
LIMIT 5;

In [0]:
# Register the raw CSVs as a temp view, with header + schema inference,
# so we can query them by name in SQL.
(spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load("/Volumes/workspace/default/citibike/202412-citibike-tripdata_*.csv")
    .createOrReplaceTempView("citibike_trips"))

# Now plain SQL against the view
result = spark.sql("""
    SELECT
        member_casual,
        COUNT(*) AS rides
    FROM citibike_trips
    GROUP BY member_casual
    ORDER BY rides DESC
""")
result.show()